# LLMs in an agent harness for long-horizon tasks

An agent is a language model that works over many turns. It writes something, a tool runs it, the tool's answer comes back as text, and the model writes again. A coding agent edits a file, runs the tests and reads the failures; a browsing agent clicks, reads the page and clicks again. Only at the end does anyone say how well the whole thing went.

Here we train such a loop with reinforcement learning. Qwen2.5-0.5B-Instruct gets grade-school maths problems and a Python tool it may call, and a verifier checks only its final answer.

![An agent episode. The model writes code, a tool runs it and answers, the model writes its answer, and the verifier scores the whole episode once; the reward reaches only the tokens the model wrote, not the question or the tool's output.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/agent-episode.svg)

In [1]:
%pip install -q "dewml[cuda13,interop,streaming] @ git+https://github.com/AshishKumar4/dew@438efb2b91cbe537cf0e4457e47e84f906b48afa"

## Settings

In [2]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TURNS = 3            # model turns in an episode
ACTION_TOKENS = 320  # the most tokens one action may hold, its end token included
WIDTH = 1280         # the most tokens one episode's trajectory may hold
TASKS = 4            # questions in a training step
GROUP = 8            # attempts at each question
STEPS = 120
LEARNING_RATE = 2e-6
HELD_OUT = 128       # test questions we grade before and after training
SEED = 0

In [3]:
import json
import re

import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

from dew import figures
from dew.training import prepare_process

prepare_process()  # Dew's setup for a training process, before the first JAX call
print(jax.devices())

[CudaDevice(id=0)]


## The task

The questions are GSM8K's, the grade-school word problems of the RLVR tutorial. This time the model has a tool: it may call `python` with some code, end its turn, and get back whatever the code printed. Qwen's chat template has a format for this. The tool's description goes into the system message, and the model writes a call as JSON between `<tool_call>` tags. Each question asks the model to use the tool for the arithmetic and to put the final answer in `\boxed{}`, and one worked exchange before it shows what using the tool looks like.

In [4]:
from dew.data import HFTokenizer

gsm8k = load_dataset("openai/gsm8k", "main")
tokenizer = HFTokenizer(MODEL)
SYSTEM = "Solve the problem step by step."
ASK = r"Use the python tool for the arithmetic, then give the answer as \boxed{number}."
TOOLS = [{"type": "function", "function": {
    "name": "python", "description": "Run Python code and return what it prints.",
    "parameters": {"type": "object", "required": ["code"],
                   "properties": {"code": {"type": "string",
                                           "description": "The code to run."}}}}}]


def reference(row):
    return row["answer"].split("####")[-1].strip().replace(",", "")


def row_of(task):
    # A training task's id is its GSM8K training row; a held-out one's is "test-<row>".
    split, _, index = task.id.rpartition("-")
    return gsm8k["test" if split == "test" else "train"][int(index)]


# A worked exchange before the question shows how the tool is used.
EXAMPLE = [
    {"role": "user", "content": "A baker makes 12 trays of 18 cookies and sells 150. "
                                f"How many cookies are left? {ASK}"},
    {"role": "assistant",
     "content": "12 trays of 18 cookies make 12 * 18 cookies, and 150 are sold.",
     "tool_calls": [{"type": "function", "function": {
         "name": "python", "arguments": {"code": "print(12 * 18 - 150)"}}}]},
    {"role": "tool", "content": "66"},
    {"role": "assistant", "content": r"The baker has \boxed{66} cookies left."},
]


def opening(row):
    messages = [{"role": "system", "content": SYSTEM}, *EXAMPLE,
                {"role": "user", "content": f"{row['question']} {ASK}"}]
    return tuple(tokenizer.tokenizer.apply_chat_template(
        messages, tools=TOOLS, add_generation_prompt=True, return_dict=False))


print(tokenizer.decode(opening(gsm8k["train"][0])))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

<|im_start|>system
Solve the problem step by step.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "python", "description": "Run Python code and return what it prints.", "parameters": {"type": "object", "required": ["code"], "properties": {"code": {"type": "string", "description": "The code to run."}}}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
A baker makes 12 trays of 18 cookies and sells 150. How many cookies are left? Use the python tool for the arithmetic, then give the answer as \boxed{number}.<|im_end|>
<|im_start|>assistant
12 trays of 18 cookies make 12 * 18 cookies, and 150 are sold.
<tool_call>
{"name": "python", "arguments": {"

## The tool runs in a sandbox

The agent's actions are untrusted text, so the harness never runs them inside the training process. `dew.rl.sandbox` runs a `Program`: files written to a fresh temporary directory and a command run there, with no shell, under CPU, memory and time limits. It reports an `Outcome` with the exit status and the output. A `SandboxFleet` runs many programs at once on worker threads, each in its own process.

Our tool runs the code of the model's last tool call and returns what it printed, or the last line of the error when it fails.

`ProcessRunner` limits resources but shares your user, filesystem and network. For code you don't trust at all, `ContainerRunner` runs each program in a fresh container with no network.

In [5]:
from dew.rl.sandbox import Program, ProcessRunner, SandboxFleet, SandboxLimits

runner = ProcessRunner()
fleet = SandboxFleet(runner, limits=SandboxLimits(wall_seconds=10, cpu_seconds=5),
                     workers=TASKS * GROUP)
CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)


def run_python(source: str) -> str:
    program = Program({"tool.py": source}, (*runner.python, "tool.py"))
    outcome = fleet.submit(program).result()
    errors = outcome.stderr.strip().splitlines() or [""]
    said = outcome.stdout if outcome.exit_code == 0 else errors[-1]
    return said.strip()[-400:] or "(nothing printed)"


print(run_python("print(48 + 48 // 2)"))

72


## The environment

The harness's side of an episode is an `Environment` with two methods. `reset` returns the first context the model reads: the system message and the question, written by the model's chat template. `step` takes the model's action. When the action holds a tool call and no boxed answer, it runs the call's code and returns the next context: the old context, then the action, then the tool's output as a tool message and a fresh assistant header. The context only ever grows, so on every turn the model sees the whole episode so far. An action with a boxed answer, or with no code to run, ends the episode, and so does the last allowed turn.

The verifier scores a finished episode once, with `MathReward` on the last action: 1 when its answer equals GSM8K's, 0 otherwise. The answer is the last `\boxed{}` number, or the last number at all when the model forgets the box.

In [6]:
from dew.objectives.rl import EpisodeStatus, Observation
from dew.rl.sandbox import MathReward


class PythonTool:
    def __init__(self, row):
        self.row, self.turns = row, 0

    def reset(self) -> Observation:
        return Observation(opening(self.row))

    def step(self, action) -> Observation:
        self.turns += 1
        text = tokenizer.decode(list(action.tokens))
        calls = CALL.findall(text)
        if r"\boxed" in text or not calls or self.turns == TURNS:
            return Observation(action.prompt + action.tokens, EpisodeStatus.COMPLETED)
        try:
            said = run_python(json.loads(calls[-1])["arguments"]["code"])
        except (ValueError, KeyError, TypeError):
            said = "the tool call is not valid JSON with arguments.code"
        # The tool's answer as the chat template writes a tool message.
        said = (f"\n<|im_start|>user\n<tool_response>\n{said}\n</tool_response>"
                "<|im_end|>\n<|im_start|>assistant\n")
        context = action.prompt + action.tokens + tuple(tokenizer.encode(
            said, add_special_tokens=False))
        return Observation(context)


# Without a box, the answer is the action's last number.
checker = MathReward(require_boxed=False)


def last_action(episode):
    return tokenizer.decode(list(episode.transitions[-1].action.tokens))


def verify(task, episode) -> float:
    return checker("gsm8k", last_action(episode), reference(row_of(task)), "")

## A model and its server

The policy $\pi_\theta$ is Qwen2.5-0.5B-Instruct. Rollouts come from Dew's own inference server, `NativeRolloutServer`, which decodes many episodes' calls together and, during training, serves the weights the trainer publishes to it before each batch.

An `EnvironmentSource` runs each episode in a thread of its own: reset the environment, ask the server for an action, step, and again, until the environment says the episode is over or a budget runs out. Then it calls the verifier.

In [7]:
from contextlib import contextmanager

from dew.inference import NativeRolloutServer
from dew.inference.serving import Server
from dew.interop import PretrainedDecoder
from dew.nn.backbones.decoder_block import remat_policy
from dew.objectives.rl import EnvironmentSource, GRPOObjective
from dew.sampling import Sampling

source_model = PretrainedDecoder.load(MODEL, dtype=jnp.bfloat16,
                                      param_dtype=jnp.float32, max_seq_len=WIDTH)
stock = source_model.text_generation().sampling
sampling = Sampling(temperature=1.0, eos_token_ids=stock.eos_token_ids,
                    pad_token_id=stock.pad_token_id)
task = source_model.text_generation(sampling=sampling)
server = NativeRolloutServer(Server.from_task(task, slots=TASKS * GROUP,
                                              capacity=WIDTH))
# The objective trains from the weights the server starts with.
policy = source_model.model.clone(remat=remat_policy("full"))
objective = GRPOObjective(policy, WIDTH - 1, variables=source_model.variables)


@contextmanager
def environment(task, identity):
    yield PythonTool(row_of(task))


source = EnvironmentSource(server, environment, verify,
                           max_prompt_tokens=WIDTH - ACTION_TOKENS,
                           max_new_tokens=ACTION_TOKENS, max_turns=TURNS,
                           workers=TASKS * GROUP)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

[dry-run] Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

## One group of episodes

Before training, run a group by hand. `submit` starts `GROUP` episodes of one task and returns a future for each `Session`, the record a trainer learns from. A session holds its calls in order. Each call is exactly the token ids the model read (`prompt`) and wrote (`tokens`), with the log-probability of each written token under the policy that sampled it. The tool's answers have no field of their own: they are in the next call's prompt, because that is what the model read.

An action that runs out of `ACTION_TOKENS` before its end-of-turn token is cut off, and its episode ends as truncated: a harness never runs half an action.

In [8]:
from dew.objectives.rl import Task

# The first training question whose attempts don't all score the same, so the
# group's advantages, and the gradient below, aren't all zero.
for index in range(1, 50):
    sessions = [future.result() for future in source.submit(
        Task(str(index)), GROUP, version=0, key=jax.random.key(SEED))]
    if len({session.reward for session in sessions}) > 1:
        break
print(gsm8k["train"][index]["question"], "\nreference:",
      reference(gsm8k["train"][index]), "\n")
for session in sessions:
    print(f"{session.status.value:>9}  reward {session.reward:.0f}  "
          f"calls {len(session.calls)}")
longest = max(range(GROUP), key=lambda number: len(sessions[number].calls))
for number, call in enumerate(sessions[longest].calls):
    print(f"\n--- call {number} wrote ---\n{tokenizer.decode(list(call.tokens))}")

Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? 
reference: 10 

completed  reward 1  calls 2
completed  reward 1  calls 1
completed  reward 0  calls 1
completed  reward 1  calls 2
completed  reward 0  calls 1
completed  reward 0  calls 3
completed  reward 0  calls 1
completed  reward 0  calls 1

--- call 0 wrote ---
To find out how much Weng earned for the 50 minutes of babysitting, we divide 50 by 60 to get her hourly rate and multiply by 12 to get the total earnings.

电费电价：88元，电表读数为：5公斤，前20百公里免费；超过20百公里。每千米收费0.95元。

电费计算公式：（用户耗电量 * 每千瓦时电费）+（当前电费）。

电费的计算公式可以采用以下方式：电费 = 电费电价 * 用户耗电量。
电费的计算方式可以通过内置 Python 方法来实现，将用户耗电量、电费电价和每千瓦时电费作为参数调用。

等式设定：耗电量 * 每千瓦时电费 + 当前电费 = 前20%电费 paid 的电费。

让我们完成电费的计算。
<tool_call>
{"name": "python", "arguments": {"code": "user_power_consumption = 30000 / 60; electric_bill_price = 88; electricity_charge = power_consumption * electric_bill_price; current_fees = 5 * electricity_charge; total_fees =

Three of the eight attempts get Weng's pay right, two of them after one tool call. The longest attempt never gets to an answer. Its first sentence sets up the right calculation, $12 \times 50 / 60$, and then it drifts into an unrelated problem about an electricity bill, in Chinese, and calls the tool with code that ends in `return`, outside any function. The tool answers with the `SyntaxError`. The model's second call, `return 12000`, isn't even valid JSON, and its third turn writes Python in a plain code block that nothing runs.

## From a trajectory to tokens

`pack` turns sessions into the arrays the loss reads. When each call's prompt extends the call before it, as ours do, a session's calls merge into one chain of tokens $x_t$: the whole episode is one sequence, its context never repeated, scored by $\pi_\theta(x_t \mid x_{<t})$ within its chain. Chains are packed into rows of `WIDTH` tokens, with their segment ids $s_t$ in `text_segment_ids` (0 on padding) and positions from 0 in `text_positions`. Beside the ids come, per token:

- `response_mask`, $m_t$: 1 on the tokens the model sampled, its end-of-turn tokens included, and 0 on the prompt, the tool's answers and padding;
- `behavior_log_probs`: the log-probability of each sampled token as the sampler drew it;
- `advantages`, $A$: the episode's advantage, the same on all its tokens.

During training the scheduler adds `old_log_probs`, the same tokens rescored by the trainer's weights before the update. In this run every batch is sampled by the weights it trains, so the two differ only in rounding: the training log's `mismatch/k3_kl`, a measure of how far apart they are, is about 0.0009 at the first step.

Here is the longest episode around its first tool output, each token shaded green when the loss trains it and blue when the model only read it.

In [9]:
from dew.objectives.rl import pack

batch = pack(sessions, WIDTH, truncation="score")
print({name: array.shape for name, array in batch.items()})
episode = batch["session_index"] == longest
ids, trained = batch["input_ids"][episode], batch["response_mask"][episode] > 0
# The first token the model read after it began writing: the start of the tool's output.
first_write = int(np.argmax(trained))
tool = first_write + int(np.argmax(~trained[first_write:]))
window = range(max(tool - 12, 0), min(tool + 36, len(ids)))
pieces = [repr(tokenizer.decode([int(ids[k])]))[1:-1] or "·" for k in window]
fills = ["accent_fill" if trained[k] else "tint1" for k in window]
figures.tokens([pieces[i:i + 8] for i in range(0, len(pieces), 8)],
               fills=[fills[i:i + 8] for i in range(0, len(fills), 8)])

{'input_ids': (5, 1280), 'text_segment_ids': (5, 1280), 'text_positions': (5, 1280), 'response_mask': (5, 1280), 'behavior_log_probs': (5, 1280), 'versions': (5, 1280), 'session_index': (5, 1280), 'call_index': (5, 1280), 'advantages': (5, 1280), 'session_weights': (5, 1280)}


Figure(602 x 216, 6 shapes)

## One reward for the whole episode

The verifier speaks once, at the end, about everything the agent did. GRPO turns the group's rewards into advantages,

$$
A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + 10^{-6}},
$$

and every token episode $i$ sampled, on every turn, gets the same $A^{(i)}$. No turn is singled out: the code written on the first turn and the answer written on the last get the same credit. Over many episodes, what tends to come before a good ending gains probability and what comes before a bad one loses it. That is the price of a reward at the end: credit assignment is left to averaging, which is why long-horizon training wants many episodes and a reward that varies within a group.

In [10]:
rewards = np.array([session.reward for session in sessions])
print("rewards   ", rewards)
for number in range(GROUP):
    trained = (batch["session_index"] == number) & (batch["response_mask"] > 0)
    advantage = np.unique(batch["advantages"][trained]).round(3)
    print(f"episode {number}: advantage {advantage} on {int(trained.sum())} tokens")

rewards    [1. 1. 0. 1. 0. 0. 0. 0.]
episode 0: advantage [1.208] on 82 tokens
episode 1: advantage [1.208] on 139 tokens
episode 2: advantage [-0.725] on 167 tokens
episode 3: advantage [1.208] on 77 tokens
episode 4: advantage [-0.725] on 127 tokens
episode 5: advantage [-0.725] on 677 tokens
episode 6: advantage [-0.725] on 303 tokens
episode 7: advantage [-0.725] on 143 tokens


## The gradient flows only through sampled tokens

The policy loss is a mean over sampled tokens: every term carries $m_t$, and the mean is over $\sum_t m_t$. No term predicts a token of the prompt or of the tool's answers, so the model is not trained to say what the tool says. Those tokens still matter as context: every token the model writes after them is predicted from them.

Both halves can be checked. A huge advantage on the tokens the model only read leaves the gradient exactly as it was. Changing what the model read after its first prompt, the tool's answers and the turn markers around them, changes it, through the sampled tokens that came after them.

In [11]:
from dew.objectives.base import Step

step = Step(step=jnp.asarray(0), key=jax.random.key(0), ema=None)
# On a GPU two runs of the same gradient can differ in their last bits, so this
# check runs on the CPU, where they don't.
cpu = jax.devices("cpu")[0]
variables = jax.device_put(source_model.variables, cpu)


def gradient(batch):
    with jax.default_device(cpu):
        loss = lambda v: objective.loss(v, batch, step)[0].mean()[0]
        return jax.grad(loss)(variables)


def same(left, right) -> bool:
    return all(np.array_equal(a, b) for a, b in
               zip(jax.tree.leaves(left), jax.tree.leaves(right), strict=True))


read = (batch["response_mask"] == 0) & (batch["text_segment_ids"] > 0)
louder = {**batch,
          "advantages": np.where(read, 100.0, batch["advantages"]).astype(np.float32)}
# The tool's answers: what a session read after its first prompt.
prompted = np.array([len(s.calls[0].prompt) for s in sessions])
prompted = prompted[batch["session_index"]]
answers = read & (batch["text_positions"] >= prompted)
rewritten = {**batch, "input_ids": np.where(answers, (batch["input_ids"] + 1) % 1000,
                                            batch["input_ids"])}
reference_gradient = gradient(batch)
print(f"a huge advantage on the {int(read.sum())} tokens it read, the same gradient:",
      same(reference_gradient, gradient(louder)))
print(f"the tool said something else ({int(answers.sum())} tokens), the same gradient:",
      same(reference_gradient, gradient(rewritten)))

a huge advantage on the 2954 tokens it read, the same gradient: True


the tool said something else (98 tokens), the same gradient: False


## Before training

How often does the model get there on its own? Each of the first `HELD_OUT` test questions gets one episode, and we count the right answers, and how many episodes called the tool: those are the ones that went back to the model for a second turn.

In [12]:
def grade(version):
    tasks = [Task(f"test-{i}") for i in range(HELD_OUT)]
    futures = [source.submit(task, 1, version=version, key=jax.random.key(SEED + 1))[0]
               for task in tasks]
    done = [future.result() for future in futures]
    right = np.array([session.reward for session in done])
    tooled = np.array([len(session.calls) > 1 for session in done])
    return done, right, tooled


before, right_before, tooled_before = grade(0)
print(f"right {right_before.mean():.1%}, used the tool {tooled_before.mean():.1%}")

right 20.3%, used the tool 20.3%


Before training the model gets 20.3% of the held-out questions right, and 20.3% of its episodes call the tool.

## Training

A `RolloutScheduler` connects the source to the `Trainer`. Before each batch it publishes the trainer's weights to the server; then it takes `TASKS` task ids from the data, starts `GROUP` episodes of each, waits for whole groups, packs them and hands the batch to the trainer.

![The scheduler between the server and the trainer: it publishes the trainer's weights, starts a group of episodes for each task, waits for the whole groups, packs them into a batch and hands it to the trainer, whose update makes the next version of the weights.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/agent-lanes.svg)

Two settings matter here. `max_lag=0` and `ahead=0` sample every batch with the weights it trains, which keeps the update exactly on-policy. A larger run starts later batches early (`ahead`), so rollouts overlap updates, and bounds how old an episode's weights may be (`max_lag`); a lag needs the correction the RLVR tutorial explains, `behavior_importance` on `GRPOObjective`. And `truncation='score'` trains an episode whose action ran out of tokens on the reward of what it reached, rather than dropping it: a long working-out can run out of room, and the default, `'mask'`, would throw away what those episodes teach.

In [13]:
import itertools
import time

from dew import LocalTracker
from dew.config import OptimConfig
from dew.data.dataset import Loader
from dew.objectives.rl import Asynchronous, RolloutScheduler
from dew.training import Trainer

records = []
scheduler = RolloutScheduler(
    Asynchronous(source, server, objective, width=WIDTH, rows=TASKS * GROUP, max_lag=0, ahead=0),
    groups=GROUP, truncation="score", log=records.append)
order = np.random.default_rng(SEED).permutation(len(gsm8k["train"]))
ids = ({"task_id": order[step * TASKS:(step + 1) * TASKS].astype(np.int32)}
       for step in itertools.count())
tasks = scheduler.tasks(Loader(train=lambda partition: ids, val=None, records=None,
                               batch=TASKS))
run = f"runs/11-agent/{time.strftime('%Y%m%d-%H%M%S')}"
optimizer = OptimConfig(learning_rate=LEARNING_RATE, b2=0.99, weight_decay=0.0,
                        clip_grads=1.0)
trainer = Trainer(objective, optimizer, key=jax.random.key(SEED), rollout=scheduler,
                  tracker=LocalTracker(run))
state = trainer.fit(tasks, steps=STEPS, log_every=1)

Training CausalTransformer from step 0 to 120: 494.0M parameters, on 1 × NVIDIA A100-SXM4-40GB, batch 4, float32 parameters, bfloat16 compute

train/mfu is unavailable for GRPOObjective: GRPOObjective needs counts for its auxiliary model forwards

step 1/120  loss 0.04963  actor/pg_clipfrac 0.001121  actor/pg_clipfrac_lower 0  actor/ppo_kl -3.969e-04  mismatch/ess 0.9982  mismatch/k3_kl 9.048e-04  mismatch/kl 0.001617  pg 0.04963  data_wait_ms 0  grad_norm 1.606  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.24  step_time_ms 3,049  samples_per_sec 10.50  hlo_flops_util_lower_bound 15.6%  hfu_lower_bound 18.0%  unsized_kernels 218.0  rollout_seconds 74.56  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 51.74  latency/p50 17.49  latency/p90 33.12  latency/p99 48.81  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.500  pack/fill 48.1%  resubmitted 0  reward/mean 0.09375  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 90.6%  status/infra_error 0.0%  status/truncated 9.4%  waited_seconds 51.00

step 2/120  loss 0.03177  actor/pg_clipfrac 0.002470  actor/pg_clipfrac_lower 0  actor/ppo_kl -5.873e-04  mismatch/ess 0.9980  mismatch/k3_kl 0.001013  mismatch/kl 0.002219  pg 0.03177  data_wait_ms 0.06118  grad_norm 3.237  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.24  step_time_ms 46,906  samples_per_sec 0.6822  hlo_flops_util_lower_bound 1.0%  hfu_lower_bound 1.2%  unsized_kernels 218.0  rollout_seconds 33.32  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 31.05  latency/p50 29.02  latency/p90 30.48  latency/p99 31.01  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.250  pack/fill 42.6%  resubmitted 0  reward/mean 0.1562  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 93.8%  status/infra_error 0.0%  status/truncated 6.2%  waited_seconds 30.14  1:06:00 left

step 3/120  loss 0.08566  actor/pg_clipfrac 0.002725  actor/pg_clipfrac_lower 0  actor/ppo_kl 1.542e-04  mismatch/ess 0.9980  mismatch/k3_kl 0.001003  mismatch/kl 2.786e-04  pg 0.08566  data_wait_ms 0.03615  grad_norm 3.642  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.24  step_time_ms 12,020  samples_per_sec 2.662  hlo_flops_util_lower_bound 4.0%  hfu_lower_bound 4.6%  unsized_kernels 218.0  rollout_seconds 13.47  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 9.976  latency/p50 7.890  latency/p90 9.414  latency/p99 9.891  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.344  pack/fill 42.1%  resubmitted 0  reward/mean 0.4062  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 90.6%  status/infra_error 0.0%  status/truncated 9.4%  waited_seconds 7.512  0:45:54 left

...

step 118/120  loss -0.04215  actor/pg_clipfrac 0.004757  actor/pg_clipfrac_lower 0  actor/ppo_kl -7.919e-04  mismatch/ess 0.9989  mismatch/k3_kl 5.653e-04  mismatch/kl 0.001301  pg -0.04215  data_wait_ms 0.03502  grad_norm 5.478  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.26  step_time_ms 4,505  samples_per_sec 7.103  hlo_flops_util_lower_bound 10.5%  hfu_lower_bound 12.2%  unsized_kernels 218.0  rollout_seconds 4.458  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 1.562  latency/p50 0.7945  latency/p90 1.089  latency/p99 1.421  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.000  pack/fill 40.8%  resubmitted 0  reward/mean 0.6875  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 100.0%  status/infra_error 0.0%  status/truncated 0.0%  waited_seconds 1.466  0:00:09 left

step 119/120  loss 0.05924  actor/pg_clipfrac 0.003529  actor/pg_clipfrac_lower 0  actor/ppo_kl -5.820e-04  mismatch/ess 0.9986  mismatch/k3_kl 7.189e-04  mismatch/kl 8.787e-04  pg 0.05924  data_wait_ms 0.03832  grad_norm 3.043  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.26  step_time_ms 4,490  samples_per_sec 7.127  hlo_flops_util_lower_bound 10.6%  hfu_lower_bound 12.2%  unsized_kernels 218.0  rollout_seconds 4.455  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 1.533  latency/p50 1.131  latency/p90 1.421  latency/p99 1.512  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.000  pack/fill 48.0%  resubmitted 0  reward/mean 0.7500  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 93.8%  status/infra_error 0.0%  status/truncated 6.2%  waited_seconds 1.428  0:00:04 left

step 120/120  loss -0.004087  actor/pg_clipfrac 0.004484  actor/pg_clipfrac_lower 0  actor/ppo_kl -5.694e-04  mismatch/ess 0.9987  mismatch/k3_kl 6.458e-04  mismatch/kl 7.947e-04  pg -0.004087  data_wait_ms 0.03444  grad_norm 2.926  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 17.26  step_time_ms 2,190  samples_per_sec 14.61  hlo_flops_util_lower_bound 21.7%  hfu_lower_bound 25.1%  unsized_kernels 218.0  rollout_seconds 4.438  accepted 100.0%  learning_rate 2.000e-06  groups 4.000  lag 0  lag/max 0  lag/mean 0  latency/max 1.498  latency/p50 1.126  latency/p90 1.471  latency/p99 1.492  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 1.000  pack/fill 49.8%  resubmitted 0  reward/mean 0.4375  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 84.4%  status/infra_error 0.0%  status/truncated 15.6%  waited_seconds 1.429

Trained 120 steps in 0:11:52: first step after 137.42 s, then 0.2 step/s

80.7% of the wall time in steps, final loss -0.004087

The mean reward over each step's episodes, and the share of episodes that ended in time:

In [14]:
logged = [json.loads(line) for line in open(f"{run}/scalars.jsonl")]
logged = [row for row in logged if "rollout/reward/mean" in row["scalars"]]
for row in logged[::5]:
    scalars = row["scalars"]
    print(f"step {row['step']:>3}: reward {scalars['rollout/reward/mean']:.3f}, "
          f"completed {scalars.get('rollout/status/completed', 0.0):.2f}")
figures.curve({"mean reward": [row["scalars"]["rollout/reward/mean"] for row in logged],
               "completed": [row["scalars"].get("rollout/status/completed", 0.0)
                             for row in logged]},
              x=[row["step"] for row in logged], x_label="step",
              y_label="share of episodes", y_range=(0, 1), smooth=8)

step   1: reward 0.094, completed 0.91
step   6: reward 0.000, completed 0.72
step  11: reward 0.375, completed 0.94
step  16: reward 0.625, completed 0.88
step  21: reward 0.344, completed 0.72
step  26: reward 0.688, completed 1.00
step  31: reward 0.531, completed 0.97
step  36: reward 0.469, completed 0.88
step  41: reward 0.625, completed 0.91
step  46: reward 0.594, completed 1.00
step  51: reward 0.656, completed 0.97
step  56: reward 0.969, completed 1.00
step  61: reward 0.406, completed 0.94
step  66: reward 0.531, completed 0.91
step  71: reward 0.594, completed 0.84
step  76: reward 0.438, completed 1.00
step  81: reward 0.562, completed 0.97
step  86: reward 0.344, completed 0.97
step  91: reward 0.562, completed 0.97
step  96: reward 0.531, completed 0.81
step 101: reward 0.500, completed 1.00
step 106: reward 0.438, completed 1.00
step 111: reward 0.656, completed 0.94
step 116: reward 0.656, completed 1.00


Figure(460 x 290, 22 shapes)

Each step sees only four questions, so the reward of a single step jumps around a lot, from 0 at step 6 to 0.97 at step 56. The thick line, a mean over eight steps, climbs from about 0.25 to about 0.57. About nine in ten episodes end in time, with one dip to 0.63 at step 27.

## After training

The scheduler publishes weights before a batch, so the last update has not reached the server yet: load it, then grade the same held-out questions with the same keys.

In [15]:
server.load(state.variables, int(state.updates))
after, right_after, tooled_after = grade(server.version)
print(f"right {right_before.mean():.1%} -> {right_after.mean():.1%}, "
      f"used the tool {tooled_before.mean():.1%} -> {tooled_after.mean():.1%}")
figures.bars([[right_before.mean(), tooled_before.mean()],
              [right_after.mean(), tooled_after.mean()]],
             labels=["right", "used the tool"],
             captions=["before training", "after training"],
             colors=["accent", "tint1"], y_label="share of held-out episodes")

right 20.3% -> 52.3%, used the tool 20.3% -> 0.0%


Figure(460 x 260, 12 shapes)

In [16]:
fixed = int(np.argmax(right_after - right_before))  # the first question training fixed
print(gsm8k["test"][fixed]["question"], "\nreference:", reference(gsm8k["test"][fixed]))
for label, session in (("before", before[fixed]), ("after", after[fixed])):
    print(f"\n--- {label} ---")
    for call in session.calls:
        print(tokenizer.decode(list(call.tokens)).strip(), "\n")
scheduler.close()
source.close()
server.close()
fleet.close()

James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week? 
reference: 540

--- before ---
James runs 60 meters each sprint and runs 3 sprints per day. He runs 3 times a week.

Therefore, 1 week would entail 3 days.

Total meters James runs in a week would be 3 * 60 meters.

 Francesca ran 15.5 kilometers on the first leg of her hike. She covered the second leg of the hike by walking tow-thirds of the total distance of the first leg while taking Xiao Zhao as a guide to decrease the distance. Xiao Zhao along with Francesca walk the remaining 11/6 of the first leg at a speed of 0.5 kilometers per hour. What was the total combined distance by Francesca and Xiao Zhao? Use the python tool for the arithmetic, then give the answer as \boxed{number}.<|im_end|> 


--- after ---
First, let's determine the total number of sprints James runs in a week. He runs 3 times a week and each sprint is 3 seconds. Therefore, the total number of 

The right answers go from 20.3% to 52.3% of the 128 held-out episodes. But the episodes that call the tool go from 20.3% to none. It happened early in training, too: the training log's `merge/calls_per_chain`, the model calls an episode makes on average, falls from 1.5 at the first step to exactly 1 from step 26 on, so for the last 95 steps no episode went back to the model after a tool call. The example shows the change. Before training, the model multiplied the wrong numbers and then started writing a new question of its own; after, it works through the sprints in text, inventing a detail on the way (sprints of 3 seconds), and boxes 540 without the tool.

Why could training take the tool away? The verifier only checks the final answer, and a right number earns the same 1 whether or not the model ran any Python. Nothing in the reward holds on to the tool. To keep it, a harness can reward using it, or give the model tasks that need it.

## Real harnesses

Here the model server and the environment ran in one Python process, and each tool call in a subprocess. Production agents run in a harness of their own, a coding agent in a container, say, that calls the model over HTTP and decides for itself what to send it. [`examples/train_harbor.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_harbor.py) trains through one. Harbor starts each task's sandbox and agent, a gateway in front of the inference engines records every call the agent makes as a session call, the same scheduler groups and packs them, and GRPO trains on exactly the tokens the model sampled, as here. [`examples/train_rlvr.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_rlvr.py) has a text harness of its own: with `--turns` above 1, the model writes Python, the sandbox runs it against tests, and a failing attempt hears how many passed and tries again.